##  09.0 Setup

In [12]:
# ===================== 09.0 Setup =====================
import sys, re, time
from pathlib import Path
from datetime import datetime
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.retriever import Retriever
from src.answer import Answerer

EVAL_DIR = ROOT / "data" / "eval"
ANSWER_QUESTIONS = EVAL_DIR / "answer_questions.csv"
# MODELS = ["llama3.2:3b", "phi4-mini"]
# MODELS = ["llama3.2:3b", "phi4-mini", "qwen3:4b"]
MODELS =["qwen3:4b"]

R = Retriever()                      # one retriever, shared by every model
print(f"{len(R.chunks)} chunks loaded")

466 chunks loaded


## 09.1 The answer test set

In [13]:
# ===================== 09.1 The answer test set =====================
# must_have / must_not: patterns separated by " && "; ALL must_have must appear, NO must_not may appear.
# kind "unanswerable": the right answer is "Not found in the guideline."
ROWS = [
 ("A01", "fact",         "When will a leaving employee get the relieving letter and full and final settlement?", r"two weeks|2 weeks", ""),
 ("A02", "fact",         "How frequently should vacancies and HR requirements be reviewed?", r"quarter", ""),
 ("A03", "table-reason", "In Exhibit 2, which selection stage is used only for service delivery staff?", r"skill test", r"computer proficiency && group discussion && written test"),
 ("A04", "fact",         "How many members should a recruitment committee have?", r"3\s*(-|to)\s*5|three to five", ""),
 ("A05", "fact",         "How long is the probation period for NHM staff?", r"three months|3 months", ""),
 ("A06", "fact",         "How many days should induction training last?", r"three to five days|3\s*(-|to)\s*5 days", ""),
 ("A07", "fact",         "What is the maximum annual increment if a State is undertaking HR rationalization?", r"15\s*%", ""),
 ("A08", "fact",         "After how many years of service can NHM staff become eligible for an experience bonus?", r"\b(3|three) years && \b(5|five)\b", ""),
 ("A09", "fact",         "How much weightage can NHM staff get when applying for jobs in the health department?", r"25\s*%", ""),
 ("A10", "fact",         "How long should the written test in the selection process be?", r"hour", ""),
 ("A11", "fact",         "Who should be on the interview panel for NHM recruitment?", r"subject matter expert && wom[ae]n", ""),
 ("A12", "fact",         "Within how many days must a grievance be sent from the facility to the district committee?", r"seven days|7 days", ""),
 ("A13", "table",        "What increment is suggested for a staff member rated 3 (satisfactory) in the appraisal?", r"5\s*%", ""),
 ("A14", "fact",         "On which day should the monthly salary be paid?", r"1st|first", ""),
 ("A15", "letter",       "Where should male MPWs be placed according to the January 2019 letter?", r"health and wellness", ""),
 ("A16", "letter",       "About how many programme management HR have been approved under NHM?", r"57,?000", ""),
 ("A17", "case-study",   "How much did Kangaroo Mother Care initiation improve under AMANAT?", r"62\.5", ""),
 ("A18", "case-study",   "For what term are contractual staff engaged in Odisha's P4P model?", r"11 months|eleven months", ""),
 ("A19", "unanswerable", "What is the salary of a District Programme Manager in Bihar?", "", ""),
 ("A20", "unanswerable", "What is the retirement age for regular government doctors in Kerala?", "", ""),
 ("A21", "unanswerable", "Which mobile app should NHM staff use to mark attendance?", "", ""),
]
if not ANSWER_QUESTIONS.exists():                     # written once; edit the CSV to add questions later
    pd.DataFrame(ROWS, columns=["qid", "kind", "question", "must_have", "must_not"]).to_csv(
        ANSWER_QUESTIONS, index=False, encoding="utf-8")

answer_qs = pd.read_csv(ANSWER_QUESTIONS, dtype=str, keep_default_na=False)
print(f"{len(answer_qs)} answer questions:", answer_qs["kind"].value_counts().to_dict())

21 answer questions: {'fact': 12, 'unanswerable': 3, 'case-study': 2, 'letter': 2, 'table-reason': 1, 'table': 1}


## 09.2 Ask every model every question

In [8]:
# ===================== 09.2 Ask every model every question =====================
rows = []
for model in MODELS:
    A = Answerer(model=model, retriever=R)
    A.generate("Say OK.")                                 # warm-up: load the model before timing
    for q in answer_qs.itertuples():
        res = A.ask(q.question)
        rows.append({"model": model, "qid": q.qid, "kind": q.kind, "question": q.question,
                     "answer": res["answer"], "not_found": res["not_found"],
                     "sources": " ".join(s["chunk_id"] for s in res["sources"]),
                     "seconds": res["seconds_search"] + res["seconds_answer"]})
        print(f"{model:12} {q.qid}  {rows[-1]['seconds']:5.1f} s")
answers_df = pd.DataFrame(rows)

d:\SwasthyaSetu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=600)

In [15]:
import time, requests
import importlib
import src.answer as answer_module

importlib.reload(answer_module)
A = answer_module  # keep A as the module interface expected by the rest of the cell
q = "Who should be in the interview panel?"
hits = R.search(q, k=5)
payload = {"model": "qwen3:4b", "stream": False, "think": False,
           "messages": [{"role": "system", "content": A.SYSTEM_PROMPT},
                        {"role": "user", "content": A.build_prompt(q, hits) + "\n/no_think"}],
           "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 512}}
t = time.time()
d = requests.post(A.OLLAMA_URL, json=payload, timeout=900).json()
print(f"total {time.time()-t:.0f}s | reading prompt {d['prompt_eval_duration']/1e9:.0f}s "
      f"({d['prompt_eval_count']} tok) | writing {d['eval_duration']/1e9:.0f}s ({d['eval_count']} tok)")
print("thinking used:", bool(d["message"].get("thinking")))
print(d["message"]["content"][:500])

total 93s | reading prompt 0s (1154 tok) | writing 91s (512 tok)
thinking used: False
Okay, I need to answer the question "Who should be in the interview panel?" using only the provided sources from the Guidelines on Human Resources for Health for NHM (2022). Let me go through each source carefully.

First, I'll check each source for information about the interview panel composition.

Source [1] (Annexure I: Suggestive Procedural Formats > Stages of Selection, page 52) mentions: "The panel should have one woman panellist and one representative of SC/ST/OBC." This is a direct stat


In [16]:
# 09.11  can prompting stop qwen3 (thinking) from reasoning out loud?
import re, time, requests
M = "qwen3:4b"
OPTS = {"temperature": 0, "num_ctx": 8192, "num_predict": 200}

STRICT = A.SYSTEM_PROMPT + ("\n8. Output ONLY the final answer. Do not explain your steps, "
                            "do not describe the sources, do not start with 'Okay' or 'Let me'.")

def chat(system, user):
    d = requests.post(A.OLLAMA_URL, timeout=900, json={
        "model": M, "stream": False, "think": False, "options": OPTS,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}).json()
    return d["message"]["content"], d["eval_count"]

def raw_empty_think(system, user):
    text = (f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n"
            "<|im_start|>assistant\n<think>\n\n</think>\n\n")          # thinking already "done"
    d = requests.post("http://localhost:11434/api/generate", timeout=900, json={
        "model": M, "stream": False, "raw": True, "prompt": text, "options": OPTS}).json()
    return d["response"], d["eval_count"]

TESTS = {
    "A_current":         lambda u: chat(A.SYSTEM_PROMPT, u),
    "B_strict":          lambda u: chat(STRICT, u),
    "C_strict+no_think": lambda u: chat(STRICT, u + "\n/no_think"),
    "D_raw_empty_think": lambda u: raw_empty_think(A.SYSTEM_PROMPT, u),
}
QS = ["Who should be in the interview panel?",
      "What is the salary of the Prime Minister of India?"]          # should be: Not found
REASONING = re.compile(r"^\s*(okay|ok\b|alright|let me|let's|first,|hmm|so,|i need|i'll)", re.I)

for q in QS:
    user = A.build_prompt(q, R.search(q, k=5))
    print("\n=====", q)
    for name, fn in TESTS.items():
        t = time.time()
        text, ntok = fn(user)
        tag = "REASONS" if REASONING.match(text) else "direct"
        print(f"--- {name}: {tag} | {ntok} tok | {time.time()-t:.0f}s\n{text.strip()[:250]}\n")


===== Who should be in the interview panel?
--- A_current: REASONS | 200 tok | 77s
Okay, the user is asking about who should be in the interview panel according to the "Guidelines on Human Resources for Health for NHM (2022)". I need to carefully check the provided sources to find the answer.

Looking at the sources, I see that Sou

--- B_strict: REASONS | 200 tok | 68s
Okay, let me tackle this question step by step. The user wants to know who should be in the interview panel according to the "Guidelines on Human Resources for Health for NHM (2022)". I need to check the provided sources carefully.

First, I'll look 

--- C_strict+no_think: REASONS | 200 tok | 37s
Okay, let me tackle this question step by step. The user is asking "Who should be in the interview panel?" based on the provided sources from the "Guidelines on Human Resources for Health for NHM (2022)". 

First, I need to carefully check each sourc

--- D_raw_empty_think: REASONS | 200 tok | 71s
Let me analyze the sources t

In [7]:
import requests
for m in MODELS:
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": m, "messages": [{"role": "user", "content": "hi"}], "stream": False})
    print(m, r.status_code, r.text[:150])

qwen3:4b 200 {"model":"qwen3:4b","created_at":"2026-10-05T12:19:28.3921803Z","message":{"role":"assistant","content":"Hello! 😊 How can I assist you today?","thinki


In [4]:
import importlib, traceback
import src.answer as A
importlib.reload(A)
from src.answer import Answerer          # re-import, so 09.2 uses the NEW class

print("URL:", A.OLLAMA_URL)
for m in MODELS:
    try:
        print(m, "->", Answerer(m, retriever=R).generate("Say hi")[:60])
    except Exception as e:
        print(m, "ERROR:", e)
        if hasattr(e, "response") and e.response is not None:
            print("   body:", e.response.text[:200])

URL: http://localhost:11434/api/chat
"qwen3:4b" ERROR: 400 Client Error: Bad Request for url: http://localhost:11434/api/chat
   body: {"error":"invalid model name"}


In [20]:
import inspect; print(inspect.getsource(A.Answerer.generate))

    def generate(self, prompt: str) -> str:
        """Send one prompt to Ollama and return its reply text."""
        payload = {
            "model": self.model,
            "messages": [{"role": "system", "content": self.system_prompt},
                         {"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0, "num_ctx": 8192},
        }
        if "qwen3" in self.model:
            payload["think"] = False          # qwen3: skip the slow thinking step
        r = requests.post(OLLAMA_URL, timeout=600, json=payload)
        r.raise_for_status()
        text = r.json()["message"]["content"]
        return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()    



## 09.3 Auto-check + scores

In [ ]:
# ===================== 09.3 Auto-check + scores =====================
patterns = lambda s: [p.strip() for p in s.split("&&") if p.strip()]
key = answer_qs.set_index("qid")

def auto_check(r):
    k = key.loc[r["qid"]]
    if r["kind"] == "unanswerable":
        return bool(r["not_found"])                       # right = it said "Not found"
    if r["not_found"]:
        return False                                      # answerable, but it gave up
    text = r["answer"].lower()
    has  = all(re.search(p, text) for p in patterns(k["must_have"]))
    bad  = any(re.search(p, text) for p in patterns(k["must_not"]))
    return has and not bad

answers_df["auto_ok"] = answers_df.apply(auto_check, axis=1)

summary = answers_df.groupby("model").agg(n=("qid", "size"), correct=("auto_ok", "mean"),
                                          avg_seconds=("seconds", "mean")).round(2)
by_kind = answers_df.pivot_table(index="kind", columns="model", values="auto_ok", aggfunc="mean").round(2)
display(summary)
display(by_kind)

# --- the failures, to read by eye ---
for r in answers_df[~answers_df["auto_ok"]].itertuples():
    print(f"{r.model} | {r.qid} [{r.kind}] {r.question}\n   → {r.answer[:300]}\n")

,n,correct,avg_seconds
model,,,
llama3.2:3b,21,0.90,30.48
phi4-mini,21,0.86,42.65


model,llama3.2:3b,phi4-mini
kind,,
case-study,1.00,1.00
fact,0.92,0.83
letter,1.00,1.00
table,1.00,1.00
table-reason,0.00,0.00
unanswerable,1.00,1.00


llama3.2:3b | A03 [table-reason] In Exhibit 2, which selection stage is used only for service delivery staff?
   → According to Exhibit 2, the selection stage used only for Service Delivery staff is:

* Written Test
* Skill Test
* Computer Proficiency
* Group Discussion
* Presentation
* Personal/ Panel Interview

All these stages are used for Service Delivery staff.

llama3.2:3b | A11 [fact] Who should be on the interview panel for NHM recruitment?
   → According to [3], the interview panel should have:

* One woman panellist
* One representative of SC/ST/OBC

[3]

phi4-mini | A03 [table-reason] In Exhibit 2, which selection stage is used only for service delivery staff?
   → Personal/Panel Interview [2]

phi4-mini | A08 [fact] After how many years of service can NHM staff become eligible for an experience bonus?
   → After 3 or 5 years of service [1][2][3]

phi4-mini | A11 [fact] Who should be on the interview panel for NHM recruitment?
   → One woman panellist and one representative 

##  09.4 Save 

In [ ]:
# ===================== 09.4 Save =====================
stamp = datetime.now().strftime("%Y%m%d_%H%M")
out = EVAL_DIR / f"answers_{stamp}.csv"
answers_df.assign(human_ok="").to_csv(out, index=False, encoding="utf-8")
print("saved", out.name, "— fill in the human_ok column (yes/no) after reading the answers")

saved answers_20261005_1453.csv — fill in the human_ok column (yes/no) after reading the answers


In [ ]:
for h in R.search("Who should be on the interview panel for NHM recruitment?", k=5):
    print(h["rank"], h["chunk_id"], h["path"][:70], h["printed_pages"])

1 L03-01 Annexure II: Compendium of Letters > Letter 3 (21st April, 2015): JS(P ['91']
2 S020-01 Chapter 5: Recruitment ['11']
3 S074-05 Annexure I: Suggestive Procedural Formats > Stages of Selection ['52']
4 S021-02 Chapter 5: Recruitment > 5.1 Bodies Conducting Recruitment ['11', '12']
5 L01-01 Annexure II: Compendium of Letters > Letter 1 (19th January, 2015): Co ['89']


In [ ]:
q = "Who should be on the interview panel for NHM recruitment?"
for m in ["bm25", "hybrid", "rerank"]:
    hits = R.search(q, k=20, method=m)
    pos = [h["rank"] for h in hits if 72 in h["pdf_pages"]]
    print(f"{m:7} page-72 chunk at rank: {pos[0] if pos else 'not in top 20'}")

bm25    page-72 chunk at rank: 1
hybrid  page-72 chunk at rank: 1
rerank  page-72 chunk at rank: 3


In [ ]:
# # 09.5  prompt candidates
# import importlib, src.answer as A
# importlib.reload(A)

# SYSTEM_V2 = f"""You answer questions about the "Guidelines on Human Resources for Health for NHM (2022)".
# How to answer:
# 1. Read ALL the numbered sources first. The answer can be in any source, not only [1].
# 2. Use ONLY these sources. No outside knowledge, no guessing.
# 3. Give the COMPLETE answer. If a source lists items (members, conditions, steps, documents), include every item.
# 4. After each fact write the number of the source you read it in, e.g. [4].
# 5. Tables: read row by row and keep each row with its column names. For "only" questions, choose rows where that column is Yes and the other columns are No.
# 6. If the sources do not answer the question, reply exactly: {A.NOT_FOUND}
# 7. Be short: 1-5 sentences or a short list. No extra advice."""

# def build_prompt_v2(question, hits):
#     """Question first (so the model knows what to look for), sources, then the question again."""
#     blocks = []
#     for n, h in enumerate(hits, 1):
#         pages = ", ".join(str(p) for p in h["printed_pages"] if p is not None) or "?"
#         kind = " (table)" if h["doc_type"] == "table" else ""
#         blocks.append(f"[{n}]{kind} {h['path']} (page {pages})\n{h['text']}")
#     return (f"Question: {question}\n\nSources:\n\n" + "\n\n".join(blocks)
#             + f"\n\nAnswer this question using the sources above: {question}\nAnswer:")

# # name -> (system prompt, prompt builder); each one adds ONE change to the one before
# VARIANTS = {
#     "v1_old":       (A.SYSTEM_PROMPT, A.build_prompt),
#     "v2_rules":     (SYSTEM_V2,       A.build_prompt),
#     "v3_rules+q1st":(SYSTEM_V2,       build_prompt_v2),
# }

In [ ]:
# # 09.6  run every variant on the same retrieved chunks
# import re, time
# import pandas as pd

# qs = pd.read_csv("../data/eval/answer_questions.csv", dtype=str).fillna("")

# def passes(answer, must_have, must_not):
#     have = [p.strip() for p in must_have.split("&&") if p.strip()]
#     bad  = [p.strip() for p in must_not.split("&&") if p.strip()]
#     return (all(re.search(p, answer, re.I) for p in have)
#             and not any(re.search(p, answer, re.I) for p in bad))

# rows = []
# for _, q in qs.iterrows():
#     hits = R.search(q["question"], k=A.TOP_K)                 # same chunks for every variant
#     for name, (system, builder) in VARIANTS.items():
#         a = A.Answerer("llama3.2:3b", retriever=R, system_prompt=system)
#         t = time.time()
#         ans = a.generate(builder(q["question"], hits))
#         rows.append({"qid": q["qid"], "kind": q["kind"], "variant": name, "answer": ans,
#                      "ok": passes(ans, q["must_have"], q["must_not"]),
#                      "cites": ",".join(sorted(set(re.findall(r"\[(\d)\]", ans)))),
#                      "sec": round(time.time() - t, 1)})
#     print(q["qid"], "done" {rows[-1]['seconds']:5.1f})

# prompt_df = pd.DataFrame(rows)
# print(prompt_df.pivot_table(index="variant", values=["ok", "sec"], aggfunc="mean").round(2))

A01 done
A02 done
A03 done
A04 done
A05 done
A06 done
A07 done
A08 done
A09 done
A10 done
A11 done
A12 done
A13 done
A14 done
A15 done
A16 done
A17 done
A18 done
A19 done
A20 done
A21 done
                 ok    sec
variant                   
v1_old         0.90  35.05
v2_rules       0.90  38.17
v3_rules+q1st  0.95  36.39


In [ ]:
# # 09.7  where variants disagree -> read these answers yourself
# wide = prompt_df.pivot(index="qid", columns="variant", values="ok")
# diff = wide[wide.nunique(axis=1) > 1]
# print(diff)
# for qid in diff.index:
#     print(f"\n===== {qid}: {qs.set_index('qid').loc[qid, 'question']}")
#     for _, r in prompt_df[prompt_df.qid == qid].iterrows():
#         print(f"--- {r.variant} (ok={r.ok}, cites={r.cites})\n{r.answer[:400]}")

variant  v1_old  v2_rules  v3_rules+q1st
qid                                     
A03       False     False           True

===== A03: In Exhibit 2, which selection stage is used only for service delivery staff?
--- v1_old (ok=False, cites=2)
According to Exhibit 2, the selection stage used only for Service Delivery staff is:

* Written Test
* Skill Test
* Computer Proficiency
* Group Discussion
* Presentation
* Personal/ Panel Interview

[2]
--- v2_rules (ok=False, cites=2)
According to Exhibit 2 in [2], the selection stage used only for Service Delivery staff is:

* Written Test: No
* Skill Test: Yes
* Computer Proficiency: No
* Group Discussion: No
* Presentation: No
* Personal/ Panel Interview: Yes

So, the selection stage used only for Service Delivery staff is Personal/ Panel Interview. [2]
--- v3_rules+q1st (ok=True, cites=2)
The selection stage used only for service delivery staff is the Skill Test [2] and the Personal/ Panel Interview [2].


In [ ]:
# # pip install tiktoken
# import tiktoken
# enc = tiktoken.get_encoding("cl100k_base")

# for text in ["hello world checking how Token works",
#              "hello world checking how token works",
#              "SwasthyaSetu dashboard"]:
#     ids = enc.encode(text)
#     pieces = [enc.decode([i]) for i in ids]
#     print(len(ids), "tokens")
#     print("  IDs   :", ids)
#     print("  pieces:", pieces)

6 tokens
  IDs   : [15339, 1917, 13598, 1268, 9857, 4375]
  pieces: ['hello', ' world', ' checking', ' how', ' Token', ' works']
6 tokens
  IDs   : [15339, 1917, 13598, 1268, 4037, 4375]
  pieces: ['hello', ' world', ' checking', ' how', ' token', ' works']
6 tokens
  IDs   : [13521, 561, 87323, 1681, 84, 28065]
  pieces: ['Sw', 'ast', 'hya', 'Set', 'u', ' dashboard']


In [ ]:
# # 09.8  do typos change what search finds?
# PAIRS = [
#     ("Can a staff nurse take maternity leave?",            "can a staf nurce take maternaty leav?"),
#     ("When will a leaving employee get the relieving letter?", "when will leaving employe get releiving leter?"),
#     ("Who should be in the interview panel?",              "who shud be in intervew pannel?"),
#     ("What is the probation period for contractual staff?","what is probaton period for contractul staf?"),
# ]
# for good, bad in PAIRS:
#     print("\n", bad)
#     for m in ["bm25", "vector", "rerank"]:
#         g = [h["chunk_id"] for h in R.search(good, 5, m)]
#         b = [h["chunk_id"] for h in R.search(bad, 5, m)]
#         print(f"  {m:7} #1 {'same' if g[0] == b[0] else 'CHANGED'} | top5 overlap {len(set(g) & set(b))}/5")


 can a staf nurce take maternaty leav?
  bm25    #1 CHANGED | top5 overlap 3/5
  vector  #1 CHANGED | top5 overlap 1/5
  rerank  #1 CHANGED | top5 overlap 1/5

 when will leaving employe get releiving leter?
  bm25    #1 CHANGED | top5 overlap 0/5
  vector  #1 CHANGED | top5 overlap 2/5
  rerank  #1 CHANGED | top5 overlap 2/5

 who shud be in intervew pannel?
  bm25    #1 CHANGED | top5 overlap 0/5
  vector  #1 CHANGED | top5 overlap 0/5
  rerank  #1 CHANGED | top5 overlap 0/5

 what is probaton period for contractul staf?
  bm25    #1 CHANGED | top5 overlap 3/5
  vector  #1 same | top5 overlap 3/5
  rerank  #1 same | top5 overlap 2/5


In [ ]:
# importlib.reload(A)
# print(A.build_prompt("test question", R.search("test question", k=2))[:300])

Question: test question

Sources:

[1] Annexure I: Suggestive Procedural Formats > Stages of Selection (page 51)
2. Skill Test

Skill Test is strongly recommended by government to ensure the quality of selected candidates. It should be conducted to assess skills that are critical for the advertised 
